In [ ]:
!pip install numpy pandas matplotlib scipy yfinance arch statsmodels -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Chart style
BLUE = '#1A3A6E'; RED = '#DC3545'; GREEN = '#2E7D32'
ORANGE = '#E67E22'; GRAY = '#666666'; PURPLE = '#8E44AD'

plt.rcParams.update({
    'figure.facecolor': 'none', 'axes.facecolor': 'none',
    'savefig.facecolor': 'none', 'savefig.transparent': True,
    'axes.grid': False, 'font.size': 10, 'axes.labelsize': 11,
    'axes.titlesize': 12, 'legend.fontsize': 9, 'xtick.labelsize': 9,
    'ytick.labelsize': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 1.0, 'axes.linewidth': 0.6,
    'legend.facecolor': 'none', 'legend.framealpha': 0, 'legend.edgecolor': 'none',
    'figure.dpi': 150,
})

def save_chart(fig, name):
    fig.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True, dpi=150)
    fig.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved: {name}')

def legend_bottom(ax, ncol=2, y=-0.18):
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)

In [ ]:
import yfinance as yf
from arch import arch_model

# Download multiple assets
tickers = ['^GSPC', 'AAPL', 'GC=F']
names = ['S&P 500', 'AAPL', 'Gold']
data = {}
for tick, name in zip(tickers, names):
    d = yf.download(tick, start='2015-01-01', end='2025-12-31', progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    data[name] = (d['Close'].pct_change() * 100).dropna()

returns_df = pd.DataFrame(data).dropna()
print(f"Joint observations: {len(returns_df)}")
print(returns_df.describe().round(3))

In [ ]:
# Step 1: Fit univariate GARCH(1,1) for each asset
garch_results = {}
std_residuals = pd.DataFrame(index=returns_df.index)

for name in names:
    m = arch_model(returns_df[name], vol='Garch', p=1, q=1, dist='t')
    r = m.fit(disp='off')
    garch_results[name] = r
    std_residuals[name] = r.resid / r.conditional_volatility
    print(f"{name}: α={r.params['alpha[1]']:.4f}, β={r.params['beta[1]']:.4f}, "
          f"persist={r.params['alpha[1]']+r.params['beta[1]']:.4f}")

In [ ]:
# Step 2: DCC estimation (simplified Engle 2002)
# Q_t = (1-a-b)*Q_bar + a*(z_{t-1} z_{t-1}') + b*Q_{t-1}
z = std_residuals.values
T, k = z.shape
Q_bar = np.corrcoef(z.T)

# Simple grid search for DCC parameters
from scipy.optimize import minimize

def dcc_loglik(params, z, Q_bar):
    a, b = params
    if a < 0 or b < 0 or a + b >= 1:
        return 1e10
    T, k = z.shape
    Q_t = Q_bar.copy()
    ll = 0
    for t in range(1, T):
        Q_t = (1 - a - b) * Q_bar + a * np.outer(z[t-1], z[t-1]) + b * Q_t
        # Normalize to correlation
        D_inv = np.diag(1.0 / np.sqrt(np.diag(Q_t)))
        R_t = D_inv @ Q_t @ D_inv
        try:
            sign, logdet = np.linalg.slogdet(R_t)
            R_inv = np.linalg.inv(R_t)
            ll += -0.5 * (logdet + z[t] @ R_inv @ z[t] - z[t] @ z[t])
        except:
            return 1e10
    return -ll

res_dcc = minimize(dcc_loglik, [0.02, 0.95], args=(z, Q_bar),
                   method='Nelder-Mead', options={'maxiter': 5000})
a_hat, b_hat = res_dcc.x
print(f"DCC parameters: a={a_hat:.4f}, b={b_hat:.4f}")
print(f"Persistence: a+b={a_hat+b_hat:.4f}")

In [ ]:
# Compute dynamic correlations
Q_t = Q_bar.copy()
dcc_corrs = np.zeros((T, k, k))
for t in range(T):
    if t > 0:
        Q_t = (1 - a_hat - b_hat) * Q_bar + a_hat * np.outer(z[t-1], z[t-1]) + b_hat * Q_t
    D_inv = np.diag(1.0 / np.sqrt(np.diag(Q_t)))
    dcc_corrs[t] = D_inv @ Q_t @ D_inv

# Plot dynamic correlations
pairs = [(0,1,'S&P 500 — AAPL'), (0,2,'S&P 500 — Gold'), (1,2,'AAPL — Gold')]
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
colors_p = [BLUE, RED, GREEN]

for ax, (i, j, label), c in zip(axes, pairs, colors_p):
    rho_t = dcc_corrs[:, i, j]
    ax.plot(returns_df.index, rho_t, color=c, lw=0.6)
    ax.axhline(Q_bar[i,j], color='black', lw=1, ls='--', alpha=0.5)
    ax.set_ylabel('ρ_t')
    ax.set_title(f'Dynamic Correlation: {label}', fontsize=10)
    ax.set_ylim(-0.5, 1.0)

axes[-1].set_xlabel('Date')
plt.suptitle('DCC-GARCH(1,1) Dynamic Correlations', fontweight='bold', y=1.01)
plt.tight_layout()
save_chart(fig, 'garch_dcc_correlations')
plt.show()